# Phase D — Streaming conformal under temporal drift (CPU)

**Goal:** Section 4.4 figure — running coverage & interval width **over time** as the
input stream drifts (in-dist → mild → severe → recover). Shows how each online method
reacts at change-points: split methods lose coverage, ACI recovers but width explodes,
**PB-JCI Online recovers coverage with stable width**.

**No GPU / no model.** Reuses `phase_C_preds_seed42.pkl` (cached inference from Phase C/Vast).
Runs in minutes on CPU.

**Attach dataset:** `sam3-q1-multiseed-ckpts` (holds `phase_C_preds_seed{42,100,200}.pkl`).

## 00 — Setup

In [ ]:
import os, glob, json, pickle, time
import numpy as np
import matplotlib.pyplot as plt
print("numpy", np.__version__)

## 01 — conformal.py (baked in)

In [ ]:
%%writefile conformal.py
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import numpy as np

def empirical_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return float("inf")
    level = np.ceil((n + 1) * (1 - alpha)) / n
    level = min(level, 1.0)
    return float(np.quantile(scores, level, method="higher"))

def pb_count(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    return (scores[:, None] * probs).sum(axis=0)

def pb_variance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    w = scores[:, None] * probs
    return (w * (1.0 - w)).sum(axis=0)

def pb_covariance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    K = probs.shape[1]
    cov = np.zeros((K, K))
    for j in range(K):
        for k in range(K):
            delta = 1.0 if j == k else 0.0
            cov[j, k] = (scores * probs[:, j] * (delta - probs[:, k])).sum()
    return cov

class MarginalSplitConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "MarginalSplitConformal":
        K = cal_gt[0].shape[0]
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                
                for k in range(K):
                    scores_per_class[k].append(float(gt[k]))
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                err = abs(gt[k] - n_pred[k]) / sigma[k]
                scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]), self.alpha)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

class AdaptiveConformalInference:
    def __init__(self, alpha_target: float = 0.1, gamma: float = 0.05):
        self.alpha_target = alpha_target
        self.gamma = gamma
        self.alpha_t = alpha_target
        self.history_q: List[float] = []
        self.history_scores: List[float] = []  

    def reset(self):
        self.alpha_t = self.alpha_target
        self.history_scores = []

    def update(self, score_t: float, covered_t: bool):
        self.history_scores.append(score_t)
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + self.gamma * (self.alpha_target - err_t)
        
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

    def get_quantile(self) -> float:
        if not self.history_scores:
            return 1.0
        return empirical_quantile(np.array(self.history_scores), self.alpha_t)

class ShiftAwareACI(AdaptiveConformalInference):
    def __init__(self, alpha_target: float = 0.1, gamma_0: float = 0.05,
                 lambda_: float = 3.0, gamma_max: float = 0.15):
        super().__init__(alpha_target, gamma_0)
        self.gamma_0 = gamma_0
        self.lambda_ = lambda_
        self.gamma_max = gamma_max
        self.gamma_t_last = gamma_0

    def update(self, score_t: float, covered_t: bool, delta_t: float = 0.0):
        self.history_scores.append(score_t)
        gamma_t = self.gamma_0 * (1.0 + self.lambda_ * max(0.0, delta_t))
        gamma_t = min(gamma_t, self.gamma_max)
        self.gamma_t_last = gamma_t
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + gamma_t * (self.alpha_target - err_t)
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

class RollingShiftDetector:
    def __init__(self, window: int = 100, cap: float = 1.0):
        self.window = window
        self.cap = cap
        self.baseline: Optional[float] = None
        self.recent: List[float] = []

    def fit_baseline(self, cal_scores) -> "RollingShiftDetector":
        self.baseline = float(np.median(np.asarray(cal_scores))) + 1e-6
        return self

    def step(self, score_t: float) -> float:
        self.recent.append(float(score_t))
        if len(self.recent) > self.window:
            self.recent.pop(0)
        cur = float(np.median(self.recent))
        delta = (cur - self.baseline) / self.baseline
        return float(np.clip(delta, 0.0, self.cap))

class PBAwareJointConformalOnline:
    def __init__(self, alpha: float = 0.1, window: int = 300):
        self.alpha = alpha
        self.window = window
        self.scores: List[float] = []

    def warmstart(self, cal_scores) -> "PBAwareJointConformalOnline":
        self.scores = list(np.asarray(cal_scores)[-self.window:])
        return self

    def get_quantile(self) -> float:
        if not self.scores:
            return float("inf")
        return empirical_quantile(np.asarray(self.scores[-self.window:]), self.alpha)

    def update(self, score_t: float):
        self.scores.append(float(score_t))
        if len(self.scores) > self.window:
            self.scores = self.scores[-self.window:]

def local_coverage_stats(covered_list, window: int = 100) -> Dict[str, float]:
    c = np.asarray(covered_list, dtype=float)
    n = len(c)
    if n == 0:
        return {"min_local_cov": 0.0, "max_miss_run": 0}
    if n >= window:
        roll = np.convolve(c, np.ones(window) / window, mode="valid")
        min_local = float(roll.min())
    else:
        min_local = float(c.mean())
    run = mx = 0
    for v in covered_list:
        run = 0 if v else run + 1
        mx = max(mx, run)
    return {"min_local_cov": min_local, "max_miss_run": int(mx)}

class PBAwareJointConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q: float = 0.0

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "PBAwareJointConformal":
        scores = []
        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            K = len(gt)
            if len(s) == 0:
                
                sigma_eps = 1.0
                S_t = max(abs(gt[k]) / sigma_eps for k in range(K))
            else:
                n_pred = pb_count(s, p)
                sigma = np.sqrt(pb_variance(s, p) + 1e-6)
                S_t = max(abs(gt[k] - n_pred[k]) / sigma[k] for k in range(K))
            scores.append(S_t)
        self.q = empirical_quantile(np.array(scores), self.alpha)
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        s = pred["scores"]
        p = pred["probs"]
        K = pred.get("K", 5)
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q * sigma)
        upper = n_pred + self.q * sigma
        return lower, upper

class ClassStratifiedConformal:
    def __init__(self, alpha: float = 0.1, bonferroni: bool = True):
        self.alpha = alpha
        self.bonferroni = bonferroni
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "ClassStratifiedConformal":
        K = cal_gt[0].shape[0]
        alpha_eff = self.alpha / K if self.bonferroni else self.alpha
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                if gt[k] > 0:  
                    err = abs(gt[k] - n_pred[k]) / sigma[k]
                    scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]) if scores_per_class[k]
                              else np.array([1.0]), alpha_eff)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

def coverage_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                       gt_counts: np.ndarray) -> np.ndarray:
    covered = (gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)
    return covered.mean(axis=0)

def joint_coverage(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                   gt_counts: np.ndarray) -> float:
    covered_all = ((gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)).all(axis=1)
    return float(covered_all.mean())

def avg_width_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> np.ndarray:
    return (intervals_hi - intervals_lo).mean(axis=0)

def macro_width(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> float:
    return float(avg_width_per_class(intervals_lo, intervals_hi).mean())

def split_calibration_test(preds: List[Dict], gts: List[np.ndarray],
                           cal_ratio: float = 0.5,
                           seed: int = 42) -> Tuple[List, List, List, List]:
    n = len(preds)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n)
    n_cal = int(n * cal_ratio)
    cal_idx = indices[:n_cal]
    test_idx = indices[n_cal:]
    cal_preds = [preds[i] for i in cal_idx]
    cal_gt = [gts[i] for i in cal_idx]
    test_preds = [preds[i] for i in test_idx]
    test_gt = [gts[i] for i in test_idx]
    return cal_preds, cal_gt, test_preds, test_gt

In [ ]:
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")
from conformal import (MarginalSplitConformal, AdaptiveConformalInference, ShiftAwareACI,
                       PBAwareJointConformal, PBAwareJointConformalOnline,
                       RollingShiftDetector, empirical_quantile, pb_count, pb_variance)
print("conformal loaded.")

## 02 — Load cached predictions

In [ ]:
WORK = "/kaggle/working"
# Prefer the multi-seed pkls (phase_C_preds_seed42/100/200.pkl); use seed 42 for the
# streaming figure (variance comes from the 5 stream seeds). Fall back to old single name.
PKL = None
for s in [42, 100, 200]:
    hits = glob.glob(f"/kaggle/input/**/phase_C_preds_seed{s}.pkl", recursive=True)
    if hits:
        PKL = hits[0]; break
if PKL is None:  # backward compat with the old single-seed pkl
    hits = glob.glob("/kaggle/input/**/phase_C_predictions.pkl", recursive=True)
    PKL = hits[0] if hits else None
assert PKL, "No phase_C_preds_seed*.pkl found - attach dataset sam3-q1-multiseed-ckpts"
print("Loading:", PKL)
with open(PKL, "rb") as f:
    d = pickle.load(f)
predictions_by_setting = d["predictions_by_setting"]
gt_counts = np.asarray(d["gt_counts"])
print("settings:", list(predictions_by_setting.keys()), "| N =", len(gt_counts))

## 03 — Build a temporal-drift stream

Stream = in_dist → mild → severe → in_dist (degrade then recover), patches sampled
per segment. Change-points let us see each method's reaction & recovery.

In [ ]:
ALPHA = 0.1
SEG_LEN = 350
ORDER = [("in_dist", SEG_LEN), ("mild_shift", SEG_LEN),
         ("severe_shift", SEG_LEN), ("in_dist", SEG_LEN)]
CHANGE_POINTS = np.cumsum([n for _, n in ORDER])[:-1]

def _score(p, gt):
    if len(p["scores"]) == 0:
        return float(abs(gt).max())
    n = pb_count(p["scores"], p["probs"]); sg = np.sqrt(pb_variance(p["scores"], p["probs"]) + 1e-6)
    return max(abs(gt[k] - n[k]) / sg[k] for k in range(len(gt)))

def _interval(p, q, K):
    if len(p["scores"]) == 0:
        return np.zeros(K), np.zeros(K)
    n = pb_count(p["scores"], p["probs"]); sg = np.sqrt(pb_variance(p["scores"], p["probs"]) + 1e-6)
    return np.maximum(0, n - q * sg), n + q * sg

def build_stream(seed):
    rng = np.random.RandomState(seed)
    N = len(gt_counts)
    pools = {s: rng.permutation(N) for s in predictions_by_setting}
    ptr = {s: 0 for s in predictions_by_setting}
    sp, sg, seg = [], [], []
    for setting, n in ORDER:
        for _ in range(n):
            i = pools[setting][ptr[setting] % N]; ptr[setting] += 1
            sp.append(predictions_by_setting[setting][i]); sg.append(gt_counts[i]); seg.append(setting)
    # calibration scores from a fresh in_dist draw (disjoint-ish)
    cal_idx = pools["in_dist"][-(N // 3):]
    cal_scores = np.array([_score(predictions_by_setting["in_dist"][i], gt_counts[i]) for i in cal_idx])
    return sp, sg, seg, cal_scores

print(f"Stream length = {sum(n for _, n in ORDER)} | change-points at {CHANGE_POINTS.tolist()}")

## 04 — Run online methods over the stream (avg over seeds)

In [ ]:
ROLL = 100
SEEDS = [0, 1, 2, 3, 4]
METHODS = ["aci", "sa_aci", "pb_jci_online", "pb_jci_split"]
mlabel = {"aci": "ACI", "sa_aci": "SA-ACI", "pb_jci_online": "PB-JCI Online",
          "pb_jci_split": "PB-JCI (split)"}

def run_stream(sp, sg, cal_scores):
    K = len(sg[0]); T = len(sp)
    covered = {m: np.zeros(T) for m in METHODS}
    width = {m: np.zeros(T) for m in METHODS}

    aci = AdaptiveConformalInference(alpha_target=ALPHA, gamma=0.05)
    aci.reset(); aci.history_scores = list(cal_scores)
    saaci = ShiftAwareACI(alpha_target=ALPHA, gamma_0=0.05, lambda_=3.0, gamma_max=0.15)
    saaci.reset(); saaci.history_scores = list(cal_scores)
    det = RollingShiftDetector(window=100).fit_baseline(cal_scores)
    pbo = PBAwareJointConformalOnline(alpha=ALPHA, window=300); pbo.warmstart(cal_scores)
    q_split = empirical_quantile(cal_scores, ALPHA)

    for t in range(T):
        p, gt = sp[t], sg[t]; s = _score(p, gt)
        for m, q in (("aci", aci.get_quantile()), ("sa_aci", saaci.get_quantile()),
                     ("pb_jci_online", pbo.get_quantile()), ("pb_jci_split", q_split)):
            lo, hi = _interval(p, q, K)
            covered[m][t] = float(((gt >= lo) & (gt <= hi)).all())
            width[m][t] = float((hi - lo).mean())
        c_aci = bool(covered["aci"][t]); aci.update(s, c_aci)
        c_sa = bool(covered["sa_aci"][t]); saaci.update(s, c_sa, delta_t=det.step(s))
        pbo.update(s)
    return covered, width

acc_cov = {m: [] for m in METHODS}
acc_wid = {m: [] for m in METHODS}
t0 = time.time()
for sd in SEEDS:
    sp, sg, seg, cal_scores = build_stream(sd)
    cov, wid = run_stream(sp, sg, cal_scores)
    for m in METHODS:
        acc_cov[m].append(cov[m]); acc_wid[m].append(wid[m])
    print(f"  seed {sd} done")
T = len(sp)
roll_cov = {m: np.convolve(np.mean(acc_cov[m], axis=0), np.ones(ROLL)/ROLL, mode="valid") for m in METHODS}
roll_wid = {m: np.convolve(np.mean(acc_wid[m], axis=0), np.ones(ROLL)/ROLL, mode="valid") for m in METHODS}
print(f"Done in {time.time()-t0:.1f}s")

## 05 — Figure: running coverage & width over time

In [ ]:
xs = np.arange(ROLL - 1, T)
colors = {"aci": "tab:blue", "sa_aci": "tab:orange",
          "pb_jci_online": "tab:green", "pb_jci_split": "tab:red"}

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

ax = axes[0]
for m in METHODS:
    ax.plot(xs, roll_cov[m] * 100, label=mlabel[m], color=colors[m], lw=1.8)
ax.axhline(90, ls="--", color="k", alpha=0.6, label="target 90%")
for cp in CHANGE_POINTS:
    ax.axvline(cp, ls=":", color="gray", alpha=0.7)
ax.set_ylabel("Running joint coverage (%)"); ax.set_ylim(0, 102)
ax.set_title("Streaming behavior under drift (in-dist -> mild -> severe -> recover)")
ax.legend(loc="lower left", ncol=3, fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
for m in METHODS:
    ax.plot(xs, roll_wid[m], label=mlabel[m], color=colors[m], lw=1.8)
for cp in CHANGE_POINTS:
    ax.axvline(cp, ls=":", color="gray", alpha=0.7)
seg_names = ["in-dist", "mild", "severe", "recover (in-dist)"]
mids = np.concatenate([[0], CHANGE_POINTS]) + SEG_LEN / 2
for mid, nm in zip(mids, seg_names):
    ax.text(mid, ax.get_ylim()[1]*0.92, nm, ha="center", fontsize=9, color="gray")
ax.set_ylabel("Running interval width"); ax.set_xlabel("stream step t")
ax.legend(loc="upper left", ncol=2, fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{WORK}/phase_D_streaming.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {WORK}/phase_D_streaming.png")

## 06 — Per-segment summary + save

In [ ]:
seg_bounds = np.concatenate([[0], CHANGE_POINTS, [T]])
seg_names = ["in_dist", "mild", "severe", "recover"]
summary = {}
print(f"{'Method':16s} | " + " | ".join(f"{s:>16s}" for s in seg_names))
print("-" * 90)
for m in METHODS:
    cov_mean = np.mean(acc_cov[m], axis=0); wid_mean = np.mean(acc_wid[m], axis=0)
    row = {}
    cells_txt = []
    for a, b, nm in zip(seg_bounds[:-1], seg_bounds[1:], seg_names):
        c = float(cov_mean[a:b].mean()) * 100; w = float(wid_mean[a:b].mean())
        row[nm] = {"coverage": c, "width": w}
        cells_txt.append(f"{c:5.1f}% / {w:6.2f}")
    summary[m] = row
    print(f"{mlabel[m]:16s} | " + " | ".join(f"{c:>16s}" for c in cells_txt))

with open(f"{WORK}/phase_D_streaming_results.json", "w") as f:
    json.dump({"config": {"alpha": ALPHA, "order": [s for s, _ in ORDER],
                          "seg_len": SEG_LEN, "seeds": SEEDS, "roll_window": ROLL},
               "per_segment": summary}, f, indent=2)
print(f"\nSaved: {WORK}/phase_D_streaming_results.json")

## Notes (Section 4.4)

- Figure shows recovery dynamics: at each change-point, split methods (PB-JCI split)
  drop coverage and stay low; ACI recovers coverage but width spikes; **PB-JCI Online**
  recovers coverage with much smaller, stable width.
- SA-ACI included to show adaptive step-size does not help (ablation, see Phase C).
- For model-seed CI on this figure, run per `phase_C_predictions_seed{s}.pkl` and
  average the curves (CPU, cheap).